# 🚀 MambaCenter: Controlled Training on Google Colab (Tesla T4)

This notebook runs the controlled evaluation of **MambaCenter** on Google Colab using a free **Tesla T4 GPU**.

### 🎯 Key Capabilities:
- **Task Switching**: Supports one-click selection between:
  - **`R1_RESUME`**: Resume DCN + Plain VSS Add-on (A1') from Epoch 8 to complete Epochs 9–15 (~9 hours).
  - **`R2_FRESH`**: Train VSS Add-on + Coarse Heatmap Aux (B1', `dlacp_34`, `use_cp_block=false`) from scratch for 15 epochs (~19 hours).
  - **`R0_BASELINE`**: Train CenterNet Baseline DLA-34 + DCNv2 for 15 epochs (~8 hours).
- **Fast Dataset Setup**: Direct Ultralytics GitHub CDN download of VisDrone2019-DET (~30 seconds) & automatic COCO conversion.
- **Hardware Acceleration**: Automatically compiles sm_75 `selective_scan_cuda` extension in ~2 minutes with Ninja.
- **Data Persistence**: Automatically connects to Google Drive so trained checkpoints (`model_last.pth`, `model_best.pth`, `history.jsonl`) are safely saved even if your Colab session disconnects.


In [ ]:
# 1. GPU Check & Compute Architecture Configuration
import os
import sys
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError("GPU is not enabled! Please go to top menu: Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU")

gpu_name = torch.cuda.get_device_name(0)
gpu_props = torch.cuda.get_device_properties(0)
vram_gb = gpu_props.total_memory / (1024**3)
major, minor = torch.cuda.get_device_capability(0)
arch_str = f"{major}.{minor}"

print(f"Active GPU:      {gpu_name} ({vram_gb:.2f} GB VRAM)")
print(f"Compute Arch:    sm_{major}{minor} (TORCH_CUDA_ARCH_LIST={arch_str})")

os.environ["TORCH_CUDA_ARCH_LIST"] = arch_str
os.environ["MAX_JOBS"] = "2"
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
!nvidia-smi


In [ ]:
# 2. Mount Google Drive for Persistent Storage
# Checkpoints will be automatically backed up here to protect against Colab disconnects.
from pathlib import Path

DRIVE_DIR = Path("/content/drive/MyDrive/MambaCenter_Checkpoints")

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    print(f"✅ Google Drive mounted! Checkpoints will persist in: {DRIVE_DIR}")
except Exception as e:
    print(f"ℹ️ Google Drive not mounted ({e}). Checkpoints will be kept in local /content storage.")


In [ ]:
# 3. Select Experiment Task
# @title 🎯 Experiment Task Selection { run: "auto" }
TASK = "R1_RESUME" # @param ["R1_RESUME", "R2_FRESH", "R0_BASELINE", "R3_CPMAMBA"]

CONFIG_MAP = {
    "R0_BASELINE": ("configs/kaggle_centernet_dla34.yaml", "centernet_colab_t4_s317", "dla_34", False),
    "R1_RESUME":   ("configs/kaggle_addon_vss.yaml", "vss_colab_t4_s317", "dlavss_34", True),
    "R2_FRESH":    ("configs/kaggle_addon_cp_b1.yaml", "cp_b1_colab_t4_s317", "dlacp_34", False),
    "R3_CPMAMBA":  ("configs/kaggle_addon_cp.yaml", "cpmamba_colab_t4_s317", "dlacp_34", False),
}

CONFIG_FILE, RUN_NAME, EXPECTED_ARCH, SHOULD_RESUME = CONFIG_MAP[TASK]
WORKDIR = Path("/content/MambaCenter")
OUT_DIR = WORKDIR / "runs" / RUN_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Selected Task:      {TASK}")
print(f"Config File:        {CONFIG_FILE}")
print(f"Run Name:           {RUN_NAME}")
print(f"Expected Arch:      {EXPECTED_ARCH}")
print(f"Resume Mode:        {SHOULD_RESUME}")
print(f"Output Directory:   {OUT_DIR}")


In [ ]:
# 4. Clone Latest Verified Codebase & Install Dependencies
import os
import sys
import shutil
from pathlib import Path

# Clone hurry211/MambaCenter repository
if not WORKDIR.exists():
    print("Cloning repository from GitHub (hurry211/MambaCenter)...")
    !git clone https://github.com/hurry211/MambaCenter.git {WORKDIR}
else:
    print(f"Codebase already present at {WORKDIR}. Pulling latest updates...")
    !cd {WORKDIR} && git pull

# Ensure CenterNet subpackages have __init__.py
for d in [WORKDIR / "baseline/CenterNet/src/lib/datasets",
          WORKDIR / "baseline/CenterNet/src/lib/models",
          WORKDIR / "baseline/CenterNet/src/lib/trains",
          WORKDIR / "baseline/CenterNet/src/lib/detectors",
          WORKDIR / "baseline/CenterNet/src/lib/utils"]:
    d.mkdir(parents=True, exist_ok=True)
    (d / "__init__.py").touch()

# Remove conflicting datasets module and install requirements
!pip uninstall -y -q datasets
!pip install -q ninja pycocotools einops timm ultralytics-thop easydict tensorboardX progress numba yacs matplotlib

# Set paths
if str(WORKDIR) not in sys.path:
    sys.path.insert(0, str(WORKDIR))
%cd {WORKDIR}
print("✅ Codebase and dependencies verified.")


In [ ]:
# 5. Compile Official selective_scan_cuda Extension (~2 minutes)
import os
import sys

print("Compiling selective_scan_cuda for Tesla T4 using Ninja...")
!python runtime/selective-scan/build.py build_ext \
  --build-lib runtime/selective-scan/packages --build-temp runtime/selective-scan/build

pkg_path = str(WORKDIR / "runtime/selective-scan/packages")
mamba_path = str(WORKDIR / "runtime/selective-scan/mamba")

for p in [pkg_path, mamba_path, str(WORKDIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

os.environ["PYTHONPATH"] = f"{pkg_path}:{mamba_path}:{WORKDIR}:" + os.environ.get("PYTHONPATH", "")

# Numerical equivalence verify
import selective_scan_cuda
from mamba_ssm.ops.selective_scan_interface import selective_scan_fn
print("✅ selective_scan_cuda compiled and imported successfully!")


In [ ]:
# 6. Download Pretrained DLA-34 Backbone Weights
import urllib.request
from pathlib import Path

hub_dir = Path("/root/.cache/torch/hub/checkpoints")
hub_dir.mkdir(parents=True, exist_ok=True)
ckpt_file = hub_dir / "dla34-ba72cf86.pth"

if not ckpt_file.exists() or ckpt_file.stat().st_size < 1000000:
    url = "https://huggingface.co/timm/dla34.in1k/resolve/main/pytorch_model.bin"
    print(f"Downloading DLA-34 ImageNet weights from {url}...")
    urllib.request.urlretrieve(url, ckpt_file)
    print(f"✅ DLA-34 ImageNet weights cached ({ckpt_file.stat().st_size / (1024**2):.2f} MB)!")
else:
    print(f"✅ DLA-34 weights already present ({ckpt_file.stat().st_size / (1024**2):.2f} MB).")


In [ ]:
# 7. Download & Convert VisDrone2019-DET Dataset (~1-2 minutes)
import json
from pathlib import Path
from PIL import Image

data_dir = WORKDIR / "data" / "visdrone"
data_dir.mkdir(parents=True, exist_ok=True)

splits = [
    ("val", "https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-val.zip", "VisDrone2019-DET-val"),
    ("train", "https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-train.zip", "VisDrone2019-DET-train")
]

for split_name, url, folder_name in splits:
    split_path = data_dir / folder_name
    zip_path = data_dir / f"{folder_name}.zip"
    if not split_path.exists():
        print(f"Downloading VisDrone {split_name} split from high-speed mirror...")
        !wget -q --show-progress -c {url} -O {zip_path}
        print(f"Extracting {zip_path}...")
        !unzip -q -o {zip_path} -d {data_dir}
        if zip_path.exists():
            zip_path.unlink()
        print(f"✅ VisDrone {split_name} ready at {split_path}")
    else:
        print(f"✅ VisDrone {split_name} already exists.")

# Convert annotations to COCO JSON format if needed
train_json = data_dir / "visdrone2019_det_train_coco.json"
val_json = data_dir / "visdrone2019_det_val_coco.json"

if not (train_json.exists() and val_json.exists()):
    print("Converting annotations via tools/visdrone2coco.py...")
    !python tools/visdrone2coco.py --root {data_dir} --splits train val
    print("✅ Annotations converted to standard COCO JSON format.")
else:
    print("✅ COCO format annotations verified.")


In [ ]:
# 8. Checkpoint Setup & Verification
import os
import glob
import shutil
import hashlib
import torch
from pathlib import Path

resume_ckpt = OUT_DIR / "model_last.pth"

if SHOULD_RESUME:
    # Look for R1 checkpoint in Drive, /content, or unzipped package
    print("Searching for authentic R1 Epoch 8 checkpoint...")
    
    zip_candidates = (
        list(Path("/content/drive/MyDrive").glob("**/r1_epoch8_checkpoint.zip")) +
        list(Path("/content").glob("**/r1_epoch8_checkpoint.zip"))
    ) if Path("/content/drive/MyDrive").exists() else list(Path("/content").glob("**/r1_epoch8_checkpoint.zip"))
    
    if zip_candidates and not resume_ckpt.exists():
        z_pkg = zip_candidates[0]
        print(f"Unpacking checkpoint bundle: {z_pkg} -> {OUT_DIR}...")
        !unzip -q -o {z_pkg} -d {OUT_DIR}
    
    # Check Drive backup directory
    drive_backup = DRIVE_DIR / RUN_NAME
    if drive_backup.exists() and not resume_ckpt.exists():
        print(f"Restoring checkpoint files from Google Drive: {drive_backup}...")
        for item in drive_backup.glob("*"):
            if item.is_file():
                shutil.copy2(item, OUT_DIR / item.name)
                print(f"  Restored {item.name}")

    # Integrity verification
    if resume_ckpt.exists():
        data = torch.load(resume_ckpt, map_location="cpu", weights_only=True)
        sha = hashlib.sha256(open(resume_ckpt, "rb").read()).hexdigest()
        print(f"\n✅ [CHECKPOINT VERIFIED]:")
        print(f"   Epoch:     {data.get('epoch')}")
        print(f"   Step:      {data.get('step')}")
        print(f"   SHA256:    {sha[:16]}...")
        assert data.get('epoch') == 8, f"Expected Epoch 8 checkpoint, but found Epoch {data.get('epoch')}"
        print("   Integrity check PASSED! Ready to resume Epochs 9-15.")
    else:
        print("\n⚠️ [NOTICE]: No R1 Epoch 8 checkpoint found yet!")
        print("   Please upload 'r1_epoch8_checkpoint.zip' to Google Drive (MyDrive) or /content, then rerun this cell.")
else:
    print(f"Clean start mode for {TASK}. Initializing fresh from DLA-34 ImageNet pretrained weights.")


In [ ]:
# 9. Main Model Training Loop (Every Epoch COCO Evaluation)
import os
import sys

TARGET_EPOCHS = 15

if SHOULD_RESUME:
    if not resume_ckpt.exists():
        raise RuntimeError("Cannot resume R1: verified model_last.pth not found! Upload r1_epoch8_checkpoint.zip first.")
    print(f"Resuming {TASK} from Epoch 8 -> Target {TARGET_EPOCHS} epochs (eval every epoch)...\n")
    train_cmd = f"{sys.executable} tools/train.py --resume --out {OUT_DIR} --set train.epochs={TARGET_EPOCHS} eval.interval=1"
else:
    print(f"Starting clean training for {TASK} -> Target {TARGET_EPOCHS} epochs (eval every epoch)...\n")
    train_cmd = f"{sys.executable} tools/train.py --config {CONFIG_FILE} --out {OUT_DIR} --set train.epochs={TARGET_EPOCHS} eval.interval=1"

print(f"Executing: {train_cmd}\n" + "="*85)
ret = os.system(train_cmd)

# Auto-backup checkpoints to Google Drive
if Path("/content/drive/MyDrive").exists():
    backup_path = DRIVE_DIR / RUN_NAME
    backup_path.mkdir(parents=True, exist_ok=True)
    print(f"\nBacking up all training outputs to Google Drive: {backup_path}...")
    for fn in ["model_best.pth", "model_last.pth", "model_final.pth", "history.jsonl", "eval_val.json", "eval_val_final.json", "log.txt", "config.yaml"]:
        f_src = OUT_DIR / fn
        if f_src.exists():
            shutil.copy2(f_src, backup_path / fn)
            print(f"  Backed up {fn} ({f_src.stat().st_size / (1024**2):.2f} MB)")
    print("✅ All artifacts securely backed up to Google Drive!")

if ret != 0:
    raise RuntimeError(f"Training loop stopped with code {ret}")


In [ ]:
# 10. Metrics Evolution Table & Plotting
import json
import matplotlib.pyplot as plt

hist_path = OUT_DIR / "history.jsonl"
if hist_path.exists():
    records = [json.loads(line) for line in hist_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    print("=" * 85)
    print(f"📊 {TASK} TRAINING PROGRESS & OFFICIAL COCO METRICS:")
    print("=" * 85)
    print(f"{'Epoch':<6} | {'Loss':<8} | {'AP@500':<10} | {'AP50@500':<10} | {'AP75@500':<10} | {'AP@100':<10} | {'img/s':<8}")
    print("-" * 85)
    
    eps, losses, ap500s, ap50s, ap75s = [], [], [], [], []
    for r in records:
        ep = r["epoch"]
        loss = r["train"].get("loss", 0.0)
        val = r.get("val", {})
        ap500 = val.get("AP@500", 0.0) * 100
        ap50 = val.get("AP50@500", 0.0) * 100
        ap75 = val.get("AP75@500", 0.0) * 100
        ap100 = val.get("AP@100", 0.0) * 100
        spd = r["train"].get("img_per_s", 0.0)
        
        eps.append(ep)
        losses.append(loss)
        ap500s.append(ap500)
        ap50s.append(ap50)
        ap75s.append(ap75)
        print(f"{ep:<6} | {loss:<8.4f} | {ap500:<10.2f}% | {ap50:<10.2f}% | {ap75:<10.2f}% | {ap100:<10.2f}% | {spd:<8.2f}")
    print("=" * 85)

    # Plot curves
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(eps, losses, 'r-o', linewidth=2, label="Train Loss")
    ax1.set_title("Training Loss Convergence")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.grid(True, linestyle="--", alpha=0.6)
    ax1.legend()

    ax2.plot(eps, ap50s, 'b-s', linewidth=2, label="AP50@500")
    ax2.plot(eps, ap500s, 'g-^', linewidth=2, label="AP@500")
    ax2.plot(eps, ap75s, 'm-d', linewidth=2, label="AP75@500")
    ax2.set_title("COCO Detection Performance")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Score (%)")
    ax2.grid(True, linestyle="--", alpha=0.6)
    ax2.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / "colab_convergence_curves.png", dpi=200)
    plt.show()
    print(f"Saved convergence curves to {OUT_DIR / 'colab_convergence_curves.png'}")
